# 76) # Ki-Kare Bağımsızlık Testi (Chi-Square Test of Independence)
Ki-Kare Uyum Testi (bir önceki konu), **tek bir kategorik değişkenin** belirli bir dağılıma uyup uymadığını test ediyordu. Ki-Kare Bağımsızlık Testi ise, **iki kategorik değişken arasında bir ilişki (bağımlılık) olup olmadığını** test eder — bu, pratikte çok daha sık karşımıza çıkacak bir test.

## Ne Zaman Kullanılır?
İki kategorik değişkenin **birbirinden bağımsız mı yoksa ilişkili mi** olduğunu merak ettiğimizde. Örnek: "Müşteri segmenti (Bireysel/Kurumsal) ile tercih edilen ödeme yöntemi (Kredi Kartı/Havale/Kapıda Ödeme) arasında bir ilişki var mı?"

## Hipotezler
- **H0:** İki değişken birbirinden **bağımsızdır** (aralarında ilişki yok)
- **H1:** İki değişken birbirine **bağımlıdır** (aralarında ilişki var)

## Kontenjans Tablosu (Contingency Table)
Bu test, verinin **çapraz tablo (cross-tabulation)** halinde düzenlenmesini gerektirir — satırlarda bir değişkenin kategorileri, 
sütunlarda diğerinin kategorileri, hücrelerde ise **frekanslar** olur.

## Test İstatistiği
Aynı Ki-Kare formülü kullanılır ($\chi^2 = \sum \frac{(O-E)^2}{E}$), ama 
"beklenen" frekanslar burada farklı hesaplanır — **eğer iki değişken 
gerçekten bağımsız olsaydı, her hücrede ne kadar frekans beklerdik** 
sorusuna göre:
$$E_{satır,sütun} = \frac{(\text{Satır Toplamı}) \times (\text{Sütun Toplamı})}{\text{Genel Toplam}}$$

## Python'da Kullanımı
```python
from scipy.stats import chi2_contingency
import pandas as pd

kontenjans_tablosu = pd.crosstab(df['Segment'], df['Odeme_Yontemi'])
chi2_stat, p_degeri, df_serbestlik, beklenen_tablo = chi2_contingency(kontenjans_tablosu)
```
Dikkat: bu fonksiyon **4 değer** döndürüyor (Uyum Testi'nden farklı) — 
`chi2_contingency` bize otomatik olarak beklenen tabloyu da hesaplayıp 
veriyor, elle hesaplamamıza gerek kalmıyor.

## Önemli Bir Varsayım
Her hücredeki **beklenen frekansın en az 5** olması önerilir — küçük örneklemlerde/çok fazla kategori olduğunda bu sağlanmayabilir, bu 
durumda Fisher's Exact Test gibi alternatifler düşünülebilir.

In [12]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
np.random.seed(42)

n = 400
segmentler = np.random.choice(['Bireysel', 'Kurumsal'], size=n, p=[0.6, 0.4])

odeme_yontemleri = []
for seg in segmentler:
    if seg == 'Bireysel':
        odeme_yontemleri.append(np.random.choice(['Kredi Karti', 'Havale', 'Kapida Odeme'], p=[0.55, 0.15, 0.30]))
    else:
        odeme_yontemleri.append(np.random.choice(['Kredi Karti', 'Havale', 'Kapida Odeme'], p=[0.30, 0.60, 0.10]))

df = pd.DataFrame({'Segment': segmentler, 'Odeme_Yontemi': odeme_yontemleri})

# H0: İki kategorik değişken arasında ilişki yoktur.
# H1: İki kategorik değişken arasında ilişki vardır. 

kontenjans = pd.crosstab(index=df['Segment'], columns=df['Odeme_Yontemi'])
statistic, pvalue, dof, expected_freq = chi2_contingency(observed=kontenjans)
print(f'χ² değeri: {statistic}')
print(f'P değeri: {pvalue}')
print(f'Serbestlik derecesi: {dof}')
print(f'Beklenen Gözlemler: {expected_freq}')
print(f'Gerçek Gözlemler: {kontenjans}')

χ² değeri: 54.88892428031469
P değeri: 1.2050956914377672e-12
Serbestlik derecesi: 2
Beklenen Gözlemler: [[ 75.7875  50.525  108.6875]
 [ 53.2125  35.475   76.3125]]
Gerçek Gözlemler: Odeme_Yontemi  Havale  Kapida Odeme  Kredi Karti
Segment                                         
Bireysel           42            65          128
Kurumsal           87            21           57


### Sonuç
Müşteri segmenti (Bireysel/Kurumsal) ile tercih edilen ödeme yöntemi 
(Havale/Kapıda Ödeme/Kredi Kartı) arasında istatistiksel olarak anlamlı 
bir ilişki olup olmadığını test etmek amacıyla Ki-Kare Bağımsızlık Testi 
uyguladık. Test sonuçlarına göre, iki değişken arasında anlamlı bir 
ilişki bulunmuştur (χ²=54.89, df=2, p<0.001).